# 🔋 Battery Paper Agent — 이차전지 논문 분석 UI

이 노트북을 **Run All** 하면 마지막 셀에서 **Gradio UI 가 인라인**으로 열립니다.

## 사용 순서
1. (최초 1회) 아래 **설치 셀** 실행
2. `.env` 파일에 `GEMINI_API_KEY` 입력 (루트의 `.env.example` 참고)
3. **Run All** → 마지막 셀에서 UI 창 열림
4. PDF 업로드(드래그&드롭 · 파일 선택 · 폴더 경로) → 모델 선택 → ▶️ **분석 시작**

## 특징
- PDF **다중 업로드** + **폴더 경로 Batch** 병행 지원
- `scripts/_models.py` 레지스트리의 **12개 Gemini 모델**을 드롭다운으로 객관식 선택
- **#10 · #11** (이름에 `-exp` 포함된 실험 모델) 선택 시 화면 상단에 `gr.Warning` 토스트 알림
- 파이프라인 Step 1~5 를 subprocess 로 순차 실행, **실시간 로그 + 진행률 표시**
- 완료 후 결과물(`.md` / `.docx`) 다운로드 링크 제공
- Jupyter 커널을 블로킹하지 않음 (웹 서버 기반)


In [ ]:
# 최초 1회만 실행 (이미 설치돼 있으면 생략 가능)
# %pip install -q -r requirements.txt


In [ ]:
# ── 1. 환경 세팅 ──
import os
import sys
import shutil
import subprocess
from pathlib import Path

from dotenv import load_dotenv
import gradio as gr

ROOT = Path().resolve()
SCRIPTS_DIR = ROOT / "scripts"
INPUT_DIR = ROOT / "input"
OUTPUT_DIR = ROOT / "output"
FINAL_REPORT_DIR = ROOT / "final_report"

for d in (INPUT_DIR, OUTPUT_DIR / "images", OUTPUT_DIR / "text", FINAL_REPORT_DIR):
    d.mkdir(parents=True, exist_ok=True)

# 공용 모델 레지스트리 로드 (scripts/_models.py)
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))
from _models import MODEL_CHOICES  # noqa: E402

load_dotenv(ROOT / ".env")

print(f"[OK] ROOT = {ROOT}")
print(f"[OK] 등록 Gemini 모델 {len(MODEL_CHOICES)}개 로드")
print(f"[OK] GEMINI_API_KEY: {'설정됨' if os.getenv('GEMINI_API_KEY') else '미설정 — .env 확인'}")

In [ ]:
# ── 2. 파이프라인 실행 로직 ──

# (script, label, needs_pdf_arg, is_critical)
PIPELINE_STEPS = [
    {"script": "01_extract.py",             "label": "📄 PDF 파싱 & 이미지 추출",     "needs_pdf": True,  "critical": True},
    {"script": "02_analyze_image.py",       "label": "🔍 Gemini Vision 이미지 해설",  "needs_pdf": False, "critical": True},
    {"script": "03_translate_summarize.py", "label": "🌐 섹션 번역 · 요약",            "needs_pdf": False, "critical": True},
    {"script": "04_assemble.py",            "label": "📝 최종 마크다운 조립",          "needs_pdf": False, "critical": True},
    {"script": "05_export_docx.py",         "label": "📎 Word(.docx) 변환 (선택)",     "needs_pdf": False, "critical": False},
]


def _parse_choice(selection: str) -> int:
    # "#3: gemini-2.5-flash" → 3
    return int(selection.split(":", 1)[0].strip().lstrip("#"))


def _safe_stem(name: str) -> str:
    return "".join(c if c.isalnum() or c in "-_." else "_" for c in name)


def _clear_intermediate():
    for p in (OUTPUT_DIR / "text", OUTPUT_DIR / "images"):
        if p.exists():
            shutil.rmtree(p)
        p.mkdir(parents=True, exist_ok=True)


def _gather_pdfs(uploaded_files, folder_path):
    pdfs: list[Path] = []
    if uploaded_files:
        for f in uploaded_files:
            raw = f.name if hasattr(f, "name") else f
            p = Path(raw)
            if p.suffix.lower() == ".pdf" and p.exists():
                pdfs.append(p)
    if folder_path:
        fp = Path(folder_path.strip().strip('"'))
        if fp.is_dir():
            pdfs.extend(sorted(fp.glob("*.pdf")))
    # Deduplicate (resolve absolute)
    seen: set[str] = set()
    unique: list[Path] = []
    for p in pdfs:
        key = str(p.resolve())
        if key not in seen:
            seen.add(key)
            unique.append(p)
    return unique


def run_pipeline(uploaded_files, folder_path, vision_sel, text_sel,
                 progress=gr.Progress(track_tqdm=False)):
    pdfs = _gather_pdfs(uploaded_files, folder_path)
    if not pdfs:
        yield "❌ 처리할 PDF 가 없습니다. 파일 업로드 또는 폴더 경로를 지정하세요.", []
        return

    vision_num = _parse_choice(vision_sel)
    text_num = _parse_choice(text_sel)

    env = os.environ.copy()
    env["GEMINI_VISION_MODEL"] = str(vision_num)
    env["GEMINI_TEXT_MODEL"] = str(text_num)
    env["PYTHONIOENCODING"] = "utf-8"  # Windows 자식 프로세스 한글 인코딩

    subprocess_kwargs = dict(
        cwd=str(ROOT),
        env=env,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    if sys.platform == "win32":
        subprocess_kwargs["creationflags"] = subprocess.CREATE_NO_WINDOW

    logs: list[str] = []
    result_paths: list[str] = []

    def emit(msg: str):
        logs.append(msg)
        return "\n".join(logs), result_paths

    yield emit(f"[START] {len(pdfs)}개 PDF 처리 | Vision=#{vision_num} | Text=#{text_num}")

    total_units = max(1, len(pdfs) * len(PIPELINE_STEPS))

    for i, pdf in enumerate(pdfs):
        safe_name = _safe_stem(pdf.stem)
        yield emit(f"\n━━━━━ [{i+1}/{len(pdfs)}] {pdf.name} ━━━━━")

        _clear_intermediate()

        # input/ 디렉토리로 복사 (이미 input/ 내부면 그대로 사용)
        input_path = INPUT_DIR / pdf.name
        if pdf.resolve() != input_path.resolve():
            shutil.copy(pdf, input_path)

        critical_failure = False
        for j, step in enumerate(PIPELINE_STEPS):
            unit = i * len(PIPELINE_STEPS) + j
            progress(unit / total_units, desc=f"[{i+1}/{len(pdfs)}] {step['label']}")
            yield emit(f"▶ [{j+1}/{len(PIPELINE_STEPS)}] {step['label']}...")

            cmd = [sys.executable, str(SCRIPTS_DIR / step["script"])]
            if step["needs_pdf"]:
                cmd.append(str(input_path))

            try:
                proc = subprocess.run(cmd, **subprocess_kwargs)
            except Exception as exc:
                yield emit(f"❌ 실행 오류: {exc}")
                if step["critical"]:
                    critical_failure = True
                    break
                continue

            if proc.stdout:
                yield emit(proc.stdout.rstrip())
            if proc.returncode != 0:
                tail = (proc.stderr or "")[-800:]
                yield emit(f"⚠️ {step['script']} 비정상 종료 (rc={proc.returncode})\n{tail}")
                if step["critical"]:
                    critical_failure = True
                    break
                yield emit("(비핵심 단계 실패 — 계속 진행)")

        # 산출물 수집 (부분 성공도 그대로 노출)
        md_src = FINAL_REPORT_DIR / "battery_paper_analysis.md"
        docx_src = FINAL_REPORT_DIR / "battery_paper_analysis.docx"
        md_dst = FINAL_REPORT_DIR / f"{safe_name}.md"
        docx_dst = FINAL_REPORT_DIR / f"{safe_name}.docx"
        if md_src.exists():
            shutil.copy(md_src, md_dst)
            result_paths.append(str(md_dst))
        if docx_src.exists():
            shutil.copy(docx_src, docx_dst)
            result_paths.append(str(docx_dst))

        if critical_failure:
            yield emit(f"❌ [{safe_name}] 중단 — 로그를 확인하세요.")
        else:
            yield emit(f"✅ [{safe_name}] 완료")

    progress(1.0, desc="완료")
    yield emit("\n🎉 전체 작업 완료")

In [ ]:
# ── 3. Gradio UI ──

EXP_WARNING_MSG = (
    "⚠️ [경고] 선택하신 모델은 실험용(Experimental) 버전입니다. "
    "구글의 정책에 따라 사전 예고 없이 API 접근이 차단되거나 응답 품질이 "
    "변동될 수 있으므로, 안정적인 작업을 원하시면 정식 버전을 선택해 주세요."
)


def _choice_labels():
    return [f"#{k}: {v}" for k, v in MODEL_CHOICES.items()]


def _default_label(num: int) -> str:
    return f"#{num}: {MODEL_CHOICES[num]}"


def _check_exp(selection: str):
    # 스펙: 이름에 "-exp" 가 포함된 경우 즉시 경고 토스트
    if selection and "-exp" in selection.lower():
        gr.Warning(EXP_WARNING_MSG)


with gr.Blocks(title="Battery Paper Agent", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🔋 이차전지 논문 분석 Agent")
    gr.Markdown("PDF 를 업로드하거나 폴더 경로를 입력하고, 모델을 고른 뒤 **분석 시작** 을 누르세요.")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 📂 입력 소스")
            pdf_input = gr.Files(
                label="PDF 파일 (드래그&드롭 또는 선택 · 다중 업로드 가능)",
                file_types=[".pdf"],
                file_count="multiple",
            )
            folder_input = gr.Textbox(
                label="또는 PDF 들이 들어있는 폴더 경로 (Batch 처리)",
                placeholder=r"예: C:\\Users\\Me\\Papers",
            )

            gr.Markdown("### 🎛️ 모델 선택 (객관식)")
            vision_dd = gr.Dropdown(
                choices=_choice_labels(),
                value=_default_label(2),
                label="🔍 Vision 모델 (Step 2 이미지 해설)",
                interactive=True,
            )
            text_dd = gr.Dropdown(
                choices=_choice_labels(),
                value=_default_label(1),
                label="📝 Text 모델 (Step 3 번역·요약)",
                interactive=True,
            )

            run_btn = gr.Button("▶️ 분석 시작", variant="primary", size="lg")

        with gr.Column(scale=2):
            gr.Markdown("### 📜 실행 로그")
            log_box = gr.Textbox(
                show_label=False,
                lines=25,
                max_lines=40,
                autoscroll=True,
                show_copy_button=True,
                interactive=False,
            )
            gr.Markdown("### 📁 결과물 다운로드")
            result_files = gr.Files(show_label=False, interactive=False)

    # 실험 모델 경고 트리거
    vision_dd.change(_check_exp, inputs=vision_dd, outputs=None)
    text_dd.change(_check_exp, inputs=text_dd, outputs=None)

    run_btn.click(
        run_pipeline,
        inputs=[pdf_input, folder_input, vision_dd, text_dd],
        outputs=[log_box, result_files],
    )

# inline=True 로 Jupyter 셀 내부에 UI 임베드. 새 탭을 원하면 inbrowser=True 로 바꾸세요.
demo.queue().launch(inline=True, share=False, quiet=True)
